# Esercizio 1 — Confronto tra metriche automatiche e valutazione umana
In base a quanto appreso durante il corso, si richiede di confrontare tre varianti/modelli (`Base`, `SFT`, `RLHF`) mediante metriche automatiche e valut umana.
---
## 1. Generazione delle risposte
Generare risposte utilizzando il modello GPT-2 tramite le 3 versioni a partire da almeno 3 prompt appartenenti a categorie differenti, ad esempio:
- Salute
- Educazione
- Coding

Per ogni prompt salvare:
- prompt originale
- risposta del modello Base
- risposta del SFT
- risposta del modello RLHF
---
## 2. Valutazione automatica
Calcolare per ciascun modello le seguenti metriche:
- BLEU
- ROUGE-1
- BERT
Utilizzare un testo di riferimento (`reference`) per il confronto con le risposte generate.
---
## 3. Confronto delle metriche
Confrontare i risultati ottenuti dai tre modelli.
Per ciascun modello riportare almeno:

| Modello |cun modello riportare almeno:
| Modello | BLEU | ROUGE-1 | BERTScore |
|----------|------|----------|------------|
| Base     |      |          |            |
| SFT      |      |          |            |
| RLHF     |      |          |            |


In [1]:
### Scrivi la tua soluzione

import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bert_score

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

# Come in lezione, SFT e RLHF sono simulati: le tre copie hanno gli stessi pesi.
modelli = {}
for nome in ["Base", "SFT", "RLHF"]:
    modelli[nome] = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
    modelli[nome].eval()
print("Base, SFT e RLHF caricati: confronto didattico, senza fine-tuning.")

esempi = [
    {
        "categoria": "Salute",
        "prompt": "Question: Why is sleep important? Answer:",
        "reference": "Sleep helps the body rest and supports memory and attention."
    },
    {
        "categoria": "Educazione",
        "prompt": "Question: What is supervised learning? Answer:",
        "reference": "Supervised learning trains a model on labeled examples to make predictions on new data."
    },
    {
        "categoria": "Coding",
        "prompt": "Question: How do you reverse a list in Python? Answer:",
        "reference": "You can reverse a list in Python with list.reverse() or create a reversed copy with list[::-1]."
    }
]

def genera_risposta(model, prompt, max_new_tokens=50):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(outputs[0, inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

def calculate_bleu(reference, candidate):
    return sentence_bleu(
        [reference.lower().split()], candidate.lower().split(),
        smoothing_function=SmoothingFunction().method1
    )

scorer = rouge_scorer.RougeScorer(["rouge1"], use_stemmer=True)
risultati = []
for esempio in esempi:
    risultato = esempio.copy()
    print("\nCategoria:", esempio["categoria"])
    print("Prompt:", esempio["prompt"])
    for nome, model in modelli.items():
        risultato[nome] = genera_risposta(model, esempio["prompt"])
        print(f"{nome}: {risultato[nome]}")
    risultati.append(risultato)

candidati = [r[nome] for r in risultati for nome in modelli]
references = [r["reference"] for r in risultati for nome in modelli]
P, R, F1 = bert_score(
    candidati, references,
    model_type="bert-base-multilingual-cased",
    lang="en", device=str(device), verbose=False
)

metriche = []
indice = 0
for risultato in risultati:
    for nome in modelli:
        candidate = risultato[nome]
        reference = risultato["reference"]
        metriche.append({
            "Categoria": risultato["categoria"],
            "Modello": nome,
            "BLEU": calculate_bleu(reference, candidate),
            "ROUGE-1": scorer.score(reference, candidate)["rouge1"].fmeasure,
            "BERTScore": F1[indice].item()
        })
        indice += 1

df_metriche = pd.DataFrame(metriche)
confronto = df_metriche.groupby("Modello")[["BLEU", "ROUGE-1", "BERTScore"]].mean()
confronto = confronto.reindex(list(modelli))
print("\nMetriche per categoria:")
print(df_metriche.round(4).to_string(index=False))
print("\nConfronto delle medie:")
print(confronto.round(4).to_string())
print("Le differenze dipendono dal campionamento, non da SFT o RLHF reali.")

cartella_risultati = Path("risultati_parte2")
cartella_risultati.mkdir(exist_ok=True)
(cartella_risultati / "risposte.json").write_text(
    json.dumps(risultati, ensure_ascii=False, indent=2), encoding="utf-8"
)
df_metriche.to_csv(cartella_risultati / "metriche_per_categoria.csv", index=False)
confronto.to_csv(cartella_risultati / "confronto_metriche.csv")

# Esempio della rubrica con gli stessi punteggi simulati usati in lezione.
# Non sono valutazioni delle risposte generate in questo esercizio.
esempio_rubrica = pd.DataFrame({
    "Modello": ["Base", "SFT", "RLHF"],
    "Cortesia": [2, 4, 5], "Informatività": [2, 3, 4],
    "Risoluzione": [1, 3, 4], "Chiarezza": [3, 4, 5], "Sicurezza": [3, 4, 5]
})
esempio_rubrica["Media"] = esempio_rubrica.iloc[:, 1:].mean(axis=1)
print("\nEsempio di valutazione umana simulata (valori della lezione):")
print(esempio_rubrica.to_string(index=False))
esempio_rubrica.to_csv(cartella_risultati / "esempio_rubrica_simulato.csv", index=False)

# La rubrica è quella della lezione; i punteggi devono essere assegnati da una persona.
criteri = ["Cortesia", "Informatività", "Risoluzione", "Chiarezza", "Sicurezza"]
scheda = []
for risultato in risultati:
    for nome in modelli:
        riga = {
            "Categoria": risultato["categoria"], "Prompt": risultato["prompt"],
            "Modello": nome, "Risposta": risultato[nome], "Annotatore": None
        }
        riga.update({criterio: None for criterio in criteri})
        scheda.append(riga)

file_valutazione = cartella_risultati / "valutazione_umana.csv"
if not file_valutazione.exists():
    pd.DataFrame(scheda).to_csv(file_valutazione, index=False)
df_umani = pd.read_csv(file_valutazione)
chiavi = ["Prompt", "Modello", "Risposta"]
df_umani = df_umani.merge(pd.DataFrame(scheda)[chiavi], on=chiavi, how="inner")
for criterio in criteri:
    df_umani[criterio] = pd.to_numeric(df_umani[criterio], errors="coerce")
annotazioni = df_umani.dropna(subset=["Annotatore"] + criteri).copy()
if not annotazioni.empty:
    if not annotazioni[criteri].isin([1, 2, 3, 4, 5]).all().all():
        raise ValueError("I punteggi della rubrica devono essere interi da 1 a 5.")
    annotazioni["Media"] = annotazioni[criteri].mean(axis=1)
    confronto_umano = annotazioni.groupby("Modello")[criteri + ["Media"]].mean()
    print("\nValutazione umana:", confronto_umano.round(2), sep="\n")
    confronto_umano.to_csv(cartella_risultati / "confronto_umano.csv")
else:
    print("\nScheda umana salvata in", file_valutazione, "(rubrica da 1 a 5).")
    print("Non sono presenti annotazioni umane reali.")


Base, SFT e RLHF caricati: confronto didattico, senza fine-tuning.

Categoria: Salute
Prompt: Question: Why is sleep important? Answer:
Base: Because it allows you to move in time. If you are in the right position and you sleep, you will get better. If you are not, you will fall asleep.

The sleep-deprived are often confused with the sleepy. The
SFT: Sleep is one of the most important physical functions. We all know that it is important to maintain our ability to remember and remember information. It is also important to keep our mind occupied and alert. It is also important to avoid excessive or excessive stress.
RLHF: Because it's a great way to get better sleep.

A: It's a great way to get better sleep.

B: It's a great way to get better sleep.

C: It's a great way to get

Categoria: Educazione
Prompt: Question: What is supervised learning? Answer:
Base: A supervised learning algorithm is a system that can be trained to learn new information about a situation or a set of conditions.

# Esercizio 2 — Valutazione della robustezza e valutazione umana

In base a quanto appreso a lezione, si richiede di valutare la capacità di tre modelli di
classificazione di gestire testi appartenenti a categorie differenti e caratterizzati dalla presenza di rumore linguistico.

#### 1. Valutazione automatica della robustezza

Utilizzando i testi presenti nel percorso `texts/`:

1. Valutare le prestazioni delle tre varianti del modello sui **testi originali/puliti**.
2. Creare, a partire dagli stessi testi, una versione **rumorosa**, introducendo errori ortografici, abbreviazioni e/o slang.
3. Valutare nuovamente le tre varianti del modello sugli stessi esempi della loro versione rumorosa.
4. Confrontare le prestazioni ottenute sui dati puliti e sui dati rumorosi.

Utilizzare come metrica principale la **macro-F1** e, facoltativamente, riportare anche l'accuracy.

Per ciascun modello riportare almeno:

| Modello   | Macro-F1 (pulito) | Macro-F1 (rumoroso) | Δ Macro-F1 |
| --------- | ----------------: | ------------------: | ---------: |
| Base      |                   |                     |            |
| SFT       |                   |                     |            |
| RLHF      |                   |                     |            |

#### 2. Analisi della perplessità per categoria

Calcolare la perplessità (Perplexity) utilizzando il modello GPT-2 separatamente per le diverse categorie presenti nei testi della cartella.

Per ciascuna categoria:

  1. Calcolare la perplessità sui testi disponibili.
  2. Confrontare i valori ottenuti tra le diverse categorie.
  3. Ordinare le categorie in base alla perplessità osservata.
  4. Discutere le possibili ragioni delle differenze, considerando caratteristiche linguistiche e strutturali dei testi.

Infine stampare a video i risultati ottenuti.


In [2]:
### Scrivi la tua soluzione

from sklearn.metrics import f1_score, accuracy_score
import re

cartella_testi = Path("texts")
categorie = {"testo 1.txt": "news", "testo 2.txt": "fiction", "testo 3.txt": "code"}
testi = [
    {"file": nome, "categoria": categoria,
     "testo": (cartella_testi / nome).read_text(encoding="utf-8")}
    for nome, categoria in categorie.items()
]

def aggiungi_rumore(testo):
    sostituzioni = {
        "nazionale": "nazzionale", "Secondo": "Secndo", "anni": "anni lol",
        "pioggia": "piogia", "città": "citta", "lettera": "lettra",
        "qualche": "qlk", "per": "x", "che": "ke", "non": "nn",
        "fibonacci": "fibonaci", "sequence": "seq", "print": "prnt"
    }
    for parola, sostituto in sostituzioni.items():
        testo = re.sub(r"\b" + re.escape(parola) + r"\b", sostituto, testo)
    return testo

for esempio in testi:
    esempio["rumoroso"] = aggiungi_rumore(esempio["testo"])

# Gli originali hanno già alcuni refusi; confronto gli stessi dati con rumore aggiuntivo.
# GPT-2 sceglie la categoria con probabilità maggiore: le tre varianti restano simulate.
def classifica_testo(model, testo):
    prompt = "Text: " + testo + "\nCategory:"
    prefisso = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    punteggi = {}
    for categoria in ["news", "fiction", "code"]:
        completamento = tokenizer(" " + categoria, return_tensors="pt").input_ids.to(device)
        input_ids = torch.cat([prefisso, completamento], dim=1)
        labels = input_ids.clone()
        labels[:, :prefisso.shape[1]] = -100
        with torch.no_grad():
            loss = model(input_ids=input_ids, labels=labels).loss
        punteggi[categoria] = -loss.item()
    return max(punteggi, key=punteggi.get)

y_true = [esempio["categoria"] for esempio in testi]
etichette = ["news", "fiction", "code"]
robustezza = []
predizioni = []
for nome, model in modelli.items():
    y_pulito = [classifica_testo(model, esempio["testo"]) for esempio in testi]
    y_rumoroso = [classifica_testo(model, esempio["rumoroso"]) for esempio in testi]
    f1_pulito = f1_score(y_true, y_pulito, labels=etichette, average="macro", zero_division=0)
    f1_rumoroso = f1_score(y_true, y_rumoroso, labels=etichette, average="macro", zero_division=0)
    robustezza.append({
        "Modello": nome, "Macro-F1 (pulito)": f1_pulito,
        "Macro-F1 (rumoroso)": f1_rumoroso, "Δ Macro-F1": f1_rumoroso - f1_pulito,
        "Accuracy (pulito)": accuracy_score(y_true, y_pulito),
        "Accuracy (rumoroso)": accuracy_score(y_true, y_rumoroso)
    })
    for esempio, pulito, rumoroso in zip(testi, y_pulito, y_rumoroso):
        predizioni.append({
            "Modello": nome, "File": esempio["file"], "Categoria vera": esempio["categoria"],
            "Predizione pulito": pulito, "Predizione rumoroso": rumoroso
        })

df_robustezza = pd.DataFrame(robustezza)
print("\nPredizioni:")
print(pd.DataFrame(predizioni).to_string(index=False))
print("\nRobustezza (delta = rumoroso - pulito):")
print(df_robustezza.round(4).to_string(index=False))
df_robustezza.to_csv(cartella_risultati / "robustezza.csv", index=False)
(cartella_risultati / "testi_puliti_rumorosi.json").write_text(
    json.dumps(testi, ensure_ascii=False, indent=2), encoding="utf-8"
)

def calculate_perplexity(model, tokenizer, text):
    input_ids = tokenizer(text, return_tensors="pt").input_ids.to(device)
    with torch.no_grad():
        outputs = model(input_ids, labels=input_ids)
    return torch.exp(outputs.loss).item()

perplessita = []
for esempio in testi:
    perplessita.append({
        "Categoria": esempio["categoria"], "File": esempio["file"],
        "Perplessità": calculate_perplexity(modelli["Base"], tokenizer, esempio["testo"])
    })
df_perplessita = pd.DataFrame(perplessita).sort_values("Perplessità")
print("\nCategorie ordinate per perplessità crescente:")
print(df_perplessita.round(2).to_string(index=False))
df_perplessita.to_csv(cartella_risultati / "perplessita.csv", index=False)
print("Un valore più basso indica un testo più prevedibile per GPT-2, non necessariamente migliore.")


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.



Predizioni:
Modello        File Categoria vera Predizione pulito Predizione rumoroso
   Base testo 1.txt           news              news                news
   Base testo 2.txt        fiction           fiction             fiction
   Base testo 3.txt           code              code                code
    SFT testo 1.txt           news              news                news
    SFT testo 2.txt        fiction           fiction             fiction
    SFT testo 3.txt           code              code                code
   RLHF testo 1.txt           news              news                news
   RLHF testo 2.txt        fiction           fiction             fiction
   RLHF testo 3.txt           code              code                code

Robustezza (delta = rumoroso - pulito):
Modello  Macro-F1 (pulito)  Macro-F1 (rumoroso)  Δ Macro-F1  Accuracy (pulito)  Accuracy (rumoroso)
   Base                1.0                  1.0         0.0                1.0                  1.0
    SFT         

Il Base ha le medie BLEU e ROUGE-1 più alte (0,0289 e 0,2710), mentre SFT ha il BERTScore maggiore (0,7131). Le metriche non bastano a garantire risposte corrette: nella risposta Coding di SFT compare `list.findall`, che non è un metodo delle liste. Le copie hanno gli stessi pesi: queste differenze dipendono dal campionamento, non dal fine-tuning. La rubrica simulata illustra il metodo della lezione; i giudizi reali vanno raccolti nella scheda.

La macro-F1 è 1,0 sia sugli originali sia sulle versioni rumorose, con delta 0 per tutte le copie. Il rumore aggiunto non ha cambiato le categorie previste, ma tre esempi non bastano per dimostrare una robustezza generale.

La perplessità cresce nell'ordine **code (9,93), news (244,44), fiction (261,97)**. Il codice contiene schemi e token ripetuti, mentre i testi italiani sono meno prevedibili per GPT-2, addestrato soprattutto in inglese. La narrativa presenta descrizioni e parole meno regolari rispetto alla notizia. Una perplessità bassa non garantisce codice corretto: il testo fornito contiene già errori.
